# AI Modified Workflow

For this workflow we use the following prompt to modify the `scale_model_size.py` notebook:

Read the workflow and come up with a summary description, put it into a README cell at the top of the notebook.

---

You can see the generated `README` in the cell below.

Overall, as a first draft, I feel that copilot did great job. The overall structure of the README seems sensible and it includes relevant details. At a cursory glance, I would be tempted to accept its results and move on with my day. That said, when I try to read it with a more critical eye I start coming up with things I'd like to change:

- The first time it uses the term "node" I think I'd want to explicitly refer to it as a "compute node" so readers don't accidentally conflate "node" with "component".
- I think this using the phrase "to characterize single-node component scaling behavior" to describe what the purpose of this workflow is could be confusing. It makes it sound like an individual component is somehow being scaled rather than that we're scaling the model size in terms of overall number of components. It's wordier but I think I'd instead say "characterize scaling on a single-node as model component size increases".
    - This whole part of the sentence is also redundant, I would remove it.
- The "Worfklow Steps" section looks accurate though it's using different labels from what I have on the headers of the notebook. Ideally I'd want there to be a 1:1 reflection here.
- There are additional details I might want to add, such as:
    - the fact that this workflow makes use of containers, and that the intent is to run it remotely on a supercomputer that uses SLURM for job management
    - that there are parameters that can be tweaked in the "global parameters cell" and a short description of what these are
    - where to find more information about SST and PHOLD
- And there's also a terminology clash between the step that extracts data from the log files.  I call this "preprocess", the README calls it "postprocessing".  How one should view it, I suppose, comes down to "what do we consider the processing step to be", is it "processing" the running of jobs (how the AI views it) or the production of the graph (how I view it).  I don't think it would be wrong, in this case, for the AI to question my terminology and ask about it, but I would prefer it to not just adopt its own the clashes with what I've already set.

When using AI to produce documentation I feel there's a real risk that it makes it "too easy" to produce something that looks good superficially, but really could benefit from the oversight of somone familiar with the context and subject matter text to critically judge, edit, and refine it.

As a consumer of documentation (and other kinds of AI-generated content too), I find this is particularly worrying. I'm afraid I'll waste my time reading something that looks superficially good but is misleading, going into irrelevant details, lacking relevant details, or just wrong.  I really wish there was some way for me to know to what degree something had human oversight.

As someone using AI-assistance I would also like to be able to tell what was AI produced and how much human oversight was applied.  As I produce AI-assisted content more frequently it will get more difficult for me to keep track of what I intended to edit, double check, or come back to.

---
---
---

# Scale Model Size Workflow

## Overview
This workflow evaluates how SST (Structural Simulation Toolkit) performance scales with increasing numbers of components on a single node. It executes the PHOLD benchmark with varying component counts (1M to 5M) and measures simulation duration to characterize single-node component scaling behavior.

## Workflow Steps
1. **Configuration** - Set up experiment parameters, container source, and benchmark repository
2. **Environment Setup** - Initialize workflow logging and prepare e4s container profile
3. **Download** - Retrieve SST container and PHOLD benchmark repository
4. **Build** - Compile the PHOLD benchmark within the container environment
5. **Execution** - Launch PHOLD simulations with increasing component sizes (1M, 2M, 3M, 4M, 5M)
6. **Monitoring** - Watch job queue status during execution
7. **Postprocessing** - Extract timing data from simulation logs and convert to CSV
8. **Visualization** - Plot simulation duration vs. component count to analyze scaling behavior

## Outputs
- `results.csv` - Extracted timing metrics and component sizes
- Scatter plot showing total simulation duration as a function of component count

# Configuration

## Import workflows module

In [ ]:
from utils.workflows import *

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our containers and benchmark results under the specified directory
# (it will be created if it doesn't already exist).
import os
if 'user_customExperimentsDir' in globals():
    baseDir = f'{user_customExperimentsDir}/scale_model_size'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/scale_model_size'

# Run using an SST in the specified container. To find containers to use see the
# container factory at https://github.com/hpc-ai-adv-dev/sst-container-factory
# Prebuild containers are available at https://github.com/orgs/hpc-ai-adv-dev/packages
container_url  = 'ghcr.io/hpc-ai-adv-dev/sst-core:master-latest'
container_name = None # DO NOT MODIFY THIS LINE: Variable will be assigned after we download the container
                      # We include it here to document what global variables are available throughout the
                      # notebook.

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Run the benchmark on a single node, increasing the numbers of components with each trial
num_comps_per_trial  = [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]

# This command will be run prior to launching a job. The command will be run
# from within the benchmark directory and execution occurs within the worklaunch
# loop so it may be parameterized by the trial parameters if needed.
prestart_cmd_template = ''

# Indicates what arguments should be passed to sst and the benchmark each run 
# Note: {width} and {height} will be replaced with the appropriate values for
# each run, based on the number of nodes and components per node
sst_args_template   = '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py'
bmark_args_template = '--width {width} --height {height}'

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'     
#   'DOWNLOAD_CONTAINERS' 
#   'DOWNLOAD_BENCHMARKS' 
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------


## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download containers

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_CONTAINERS' in force

container_name = download_custom_container(container_url, force=_force)

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/{benchmarkPath}")
run_cmd('touch sstsimulator.conf')
_cmd = 'make' if not _force else 'make clean; make'
run_in_container(_cmd,
    f'{baseDir}/{container_name}',
    additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
cd(baseDir)

# Run

## Start jobs

In [ ]:
import math, shutil, os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Setup directory to store results in
run_dir = f'{baseDir}/runs/'
if os.path.exists(run_dir):
    shutil.rmtree(run_dir)
os.makedirs(run_dir, exist_ok=True)

cd(f"{baseDir}/benchmarks/{benchmarkPath}")

# Deploy jobs
for approx_size in num_comps_per_trial:
    width  = int(math.sqrt(approx_size))
    height = width
    size = width*height

    if prestart_cmd_template is not None and prestart_cmd_template != '':
        run_cmd(prestart_cmd_template.format(width=width, height=height, size=size))

    full_sst_args_template = f'{sst_args_template} -- {bmark_args_template}'
    sst_args = full_sst_args_template.format(width=width, height=height, size=size)

    launch_and_log_sst(
        image        = f'{baseDir}/{container_name}',
        srun_args    = f'-N 1 --job-name={benchmarkPath.lower()}_{size} {additional_srun_args}',
        sst_args     = sst_args,
        log_file     = f'{run_dir}/size_{size}',
        config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
        safe_display = runDisplay)

cd(f"{baseDir}")

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [ ]:
inspect_logs(f'{baseDir}/runs')

# Preprocess

In [ ]:
import os, glob

fullpath = f"{baseDir}/runs"
print(f'\n===== running extract under {fullpath} =====')
cd(fullpath)

data = extract_sst_output_in_files(sorted(glob.glob("size_*")))
csv_lines = convert_to_csv(data)
csv_name = f"{baseDir}/runs/results.csv"

with open(csv_name, 'w') as f:
    f.write('\n'.join(csv_lines))

if os.path.exists(csv_name):
    with open(csv_name, 'r') as f:
        print(f'\n===== {csv_name} =====')
        print(f.read())
else:
    print(f'\n===== {csv_name} (not created) =====')

# Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    df = pd.read_csv(f"{baseDir}/runs/results.csv")
except FileNotFoundError as e:
    print(f'ERROR: File not found - {e.filename}')
    raise StopExecution()

fig = plt.figure()
ax = fig.add_subplot(111)

plot_value='total_duration'
ylabel = 'Total duration (secs)'

ax.scatter(x=df["Size"], y=df[plot_value], c='b', marker="s", label=f'SST 15.1.0')
ax.legend().remove()
plt.title(f'SST {benchmarkPath} single-node component scaling ({plot_value})')
plt.xlabel('Number of components')
plt.ylabel(ylabel)
plt.show()